# 02 - Preprocessing & Feature Engineering
**Goal:** Clean the data, encode categorical variables, scale numerical features and preapre a model ready dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

print("Libraries Loaded")

Libraries Loaded


#### Loading Data from EDA

In [2]:
df = pd.read_csv('../data/telco_after_eda.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (7032, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df = df.drop(columns=['customerID'])
print("Dropped customerID")

Dropped customerID


#### Encode Target

In [4]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())

Churn
0    5163
1    1869
Name: count, dtype: int64


#### Encoding All Categorical Columns at once

In [5]:
# Get all remaining string columns
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

# Binary columns (2 unique values) - Label Encode
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
print(f"\nBinary cols: {binary_cols}")

for col in binary_cols:
    df[col] = (df[col] == df[col].unique()[0]).astype(int)

# Multi-class columns (3+ unique values) - One Hot Encode
multi_cols = [col for col in cat_cols if df[col].nunique() > 2]
print(f"Multi-class cols: {multi_cols}")

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

# Final check — no strings should remain
remaining = df.select_dtypes(include='str').columns.tolist()
print(f"\nString columns still remaining: {remaining}")
print("All clear!" if len(remaining) == 0 else "Fix these columns!")

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Binary cols: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi-class cols: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

String columns still remaining: []
All clear!


#### Scale Numerical Columns

In [6]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

with open('../src/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Scaled and scaler saved")

Scaled and scaler saved


#### Save Feature Columns

In [7]:
feature_cols = [col for col in df.columns if col != 'Churn']

with open('../src/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print(f"Feature columns saved: {len(feature_cols)} features")

Feature columns saved: 30 features


#### Final Check

In [8]:
print(f"Final shape: {df.shape}")
print(f"Any nulls: {df.isnull().sum().sum()}")
print(f"Dtypes remaining:\n{df.dtypes.value_counts()}")

Final shape: (7032, 31)
Any nulls: 0
Dtypes remaining:
bool       21
int64       7
float64     3
Name: count, dtype: int64


#### Saving PReprocessed Data

In [9]:
df.to_csv('../data/telco_preprocessed.csv', index=False)
print("Saved telco_preprocessed.csv ✅")

Saved telco_preprocessed.csv ✅


## Preprocessing Summary

1. Dropped `customerID` - not a predictive feature
2. Encoded `Churn` as binary 0/1
3. Label encoded binary categorical columns
4. One-hot encoded multi-class categorical columns
5. Standard scaled numerical columns (tenure, MonthlyCharges, TotalCharges)
6. Saved scaler and feature columns for Streamlit app consistency

**Output:** `telco_preprocessed.csv` ready for modeling